Import libraries : Pandas, s3fs

In [ ]:
from contextlib import nullcontext
from tarfile import NUL

import pandas as pd


From https://www.kaggle.com/datasets/mmmarchetti/tweets-dataset,download , download the csv file and load to the directory

Perform Error Handling to ensure files loaded to the Jupiter Notebook

In [ ]:
try :
        csv_file = "tweets.csv"
        df = pd.read_csv(csv_file)
        print(df) # note: with DataFrame not required to include print to df.head(2). Otherwise, Yes!
except FileNotFoundError as e:
    print(e,"Check whether file is loaded to the directory")
except Exception as e:
    print("Some other error occurred", e)
finally:
    print("The aim is to load and read the csv file named tweets.csv")

Check whether columns are NaN 

In [ ]:
find_country_name = df["country"].notna()
df_countries = df[find_country_name]
df_countries

In [ ]:
find_latitude = df["content"].notna()
df_latitude = df[find_latitude]
df_latitude

In [ ]:
find_language = df["language"] !="en"
df_language = df[find_language]
df_language

Check whether latitude mean can be calculated to replace the NaN 

In [ ]:
df["latitude"].mean()

In [ ]:
df["longitude"].mean()


In [ ]:
df["country"].unique()


Drop the columns

In [ ]:
df.drop(["latitude","longitude"],axis=1)


In [ ]:
df_cleaned_csv1= df.drop(["country","latitude","longitude"],axis=1,inplace =True)


In [ ]:
df


Calculate the total engagement by combining likes and share for each author

In [ ]:
df_likes_and_share = df.groupby("author").aggregate({"number_of_likes":"sum","number_of_shares":"sum"})

df_likes_and_share
df_likes_and_share.sort_values("number_of_likes",ascending=False)

In [ ]:
df_grouped = df.groupby("author").aggregate({"number_of_likes":"sum","number_of_shares":"sum"})
df_grouped["number_of_likes"]

In [ ]:
df_grouped.sort_values(["number_of_likes","number_of_shares"],ascending=False)
df_grouped

In [ ]:
df_grouped["total_engagement"] = (
    df_grouped["number_of_likes"] + df_grouped["number_of_shares"]
)
df_grouped

Save the total engagement to csv

In [ ]:
df_grouped.to_csv("author_engagement.csv",index=True)


Download Packages to perform sentimental analysis on tweets made by authors

In [ ]:
pip install nltk

In [ ]:
import nltk

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

To handle ssl contents

In [ ]:
import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('vader_lexicon')

In [ ]:
sia = SentimentIntensityAnalyzer()


Test a sample using sia.

In [ ]:
text = "I love this phone, it's amazing!"

print(sia.polarity_scores(text))

Function to calculate the sentiment score for each tweet

In [ ]:
def sentiment_analyser(text):
    score = sia.polarity_scores(text)["compound"]
    if score >=0.5:
        return 1
    elif score <= 0.5:
        return 0
    else:
        return 0.5

df["sentiment_score"] = df["content"].apply(sentiment_analyser)

In [ ]:
df["sentiment_score"]


Save the sentiment score to csv

In [ ]:
df.to_csv("sentimental_analyser",index=True)

In [ ]:
df.drop(["latitude","longitude","country"],axis=1,inplace=True)

In [ ]:
df_author_sentimental_score = df.groupby("author")["sentiment_score"].sum().reset_index()
df_author_sentimental_score

In [ ]:
df_author_sentimental_score.to_csv("author_sentiment_score.csv", index=False)